In [ ]:
import requests
import json
import pandas as pd
import os
import tqdm as tqdm
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus


# Define your API key
api_key = 'ntioungsta'

In [ ]:
# Load dataset
df = pd.read_csv('/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/europeana_datasets - europeana_datasets.csv')
df.head()

In [ ]:
# get last 500 records
df_firsts = df[:1292]
total_count = df_firsts['total_count'].sum()
print('Total:', total_count)

df1 = df_firsts[:10]
df2 = df_firsts[10:50]
df3 = df_firsts[50:100]
df4 = df_firsts[100:200]
df5 = df_firsts[200:400]
df6 = df_firsts[400:600]
df7 = df_firsts[600:800]
df8 = df_firsts[800:1000]
df9 = df_firsts[1000:1292]
df10 = df[1292:]

In [ ]:
df = df7

# Define the number of documents to fetch
N = 100000

# Calculate the number of documents to fetch from each dataset
num_datasets = len(df)
documents_per_dataset = max(1, N // num_datasets)

# Define the number of rows per request
rows = 100

# Initialize list to store document IDs and documents
document_ids = []
documents = []

# Set a limit for the maximum number of requests per dataset
max_requests = 100  # Adjust based on how many documents you need

print(f"Number of documents to fetch: {N}")
print(f"Number of documents per dataset: {documents_per_dataset}")
print(f"Number of datasets: {num_datasets}")
print(f"Number of requests per dataset: {max_requests}")

# Function to get documents using cursor-based pagination
def get_documents(api_key, dataset_name, rows, cursor):
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&cursor={cursor}'
    response = requests.get(search_url)
    
    if response.status_code == 429:  # Rate limit exceeded
        print("Rate limit exceeded. Waiting before retrying...")
        time.sleep(60)  # Wait for 60 seconds before retrying
        return [], cursor

    data = response.json()
    # Check if 'items' key is in the response
    if 'items' in data:
        document_ids = [item['id'] for item in data['items']]
    else:
        document_ids = []
    
    next_cursor = data.get('nextCursor', None)
    safe_next_cursor = quote_plus(next_cursor) if next_cursor else None
    return document_ids, safe_next_cursor

# Function to fetch document by ID
def fetch_document(api_key, doc_id):
    record_url = f'https://api.europeana.eu/record/v2/{doc_id}.rdf?wskey={api_key}'
    response = requests.get(record_url)
    
    if response.status_code == 429:  # Rate limit exceeded
        print(f"Rate limit exceeded while fetching document {doc_id}. Waiting before retrying...")
        time.sleep(60)  # Wait for 60 seconds before retrying
        return None

    return response.text

# Loop through each dataset to collect documents
for index, row in df.iterrows():
    dataset_name = row[0]
    cursor = '*'
    current_request = 0
    dataset_documents = []

    print(f'Collecting from dataset: {dataset_name}')
    
    while (cursor and current_request < max_requests):
        print('number of documents from this dataset:', len(dataset_documents))
        print(f'Request #{current_request + 1} for dataset {dataset_name}')
        current_request += 1
        new_document_ids, cursor = get_documents(api_key, dataset_name, rows, cursor)
        
        document_ids.extend(new_document_ids)
        print('total number of documents:', len(document_ids))

        # Fetch each document by its ID using concurrent requests
        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_doc = {executor.submit(fetch_document, api_key, doc_id): doc_id for doc_id in new_document_ids}
            for future in as_completed(future_to_doc):
                try:
                    result = future.result()
                    if result:
                        documents.append(result)
                        dataset_documents.append(result)
                    if len(documents) >= N:
                        break
                except Exception as e:
                    print(f"An error occurred: {e}")
        
        if len(dataset_documents) >= documents_per_dataset:
            break
        
        # To avoid hitting the API rate limit
        time.sleep(0.05)  # Adjust sleep time as needed
        
        if len(documents) >= N:
            break

    if len(documents) >= N:
        break

In [ ]:
# Check the number of documents collected
print(f'Total documents collected: {len(documents)}')

In [ ]:
def save_documents(dataset_name, documents):
    if not os.path.exists(dataset_name):
        os.makedirs(dataset_name)

    for i, doc in enumerate(documents):
        with open(f'{dataset_name}/document_{i}.rdf', 'w') as f:
            f.write(doc)

save_documents("hunnid_7", documents)

## Other versoin

In [ ]:
df = df2

# Define the number of documents to fetch
N = 100000

# Calculate the number of documents to fetch from each dataset
num_datasets = len(df)
documents_per_dataset = max(1, N // num_datasets)
print(f"Number of documents per dataset: {documents_per_dataset}")

# Define the number of rows per request
rows = 100

# Initialize list to store document IDs and documents
document_ids = []
documents = []

# Set a limit for the maximum number of requests per dataset
max_requests = 100  # Adjust based on how many documents you need

# Function to get documents using cursor-based pagination
def get_documents(api_key, dataset_name, rows, cursor):
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&cursor={cursor}'
    response = requests.get(search_url)
    data = response.json()
    # Check if 'items' key is in the response
    if 'items' in data:
        document_ids = [item['id'] for item in data['items']]
    else:
        document_ids = []
    
    next_cursor = data.get('nextCursor', None)
    safe_next_cursor = quote_plus(next_cursor) if next_cursor else None
    return document_ids, safe_next_cursor

# Function to fetch document by ID
def fetch_document(api_key, doc_id):
    record_url = f'https://api.europeana.eu/record/v2/{doc_id}.rdf?wskey={api_key}'
    response = requests.get(record_url)
    return response.text

# Loop through each dataset to collect documents
for index, row in df.iterrows():
    dataset_name = row[0]
    cursor = '*'
    current_request = 0
    dataset_documents = []

    print(f'Collecting from dataset: {dataset_name}')
    
    while cursor and current_request < max_requests or len(dataset_documents) < documents_per_dataset:
        print(f'Request #{current_request + 1} for dataset {dataset_name}')
        current_request += 1
        new_document_ids, cursor = get_documents(api_key, dataset_name, rows, cursor)
        
        document_ids.extend(new_document_ids)

        # Fetch each document by its ID using concurrent requests
        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_doc = {executor.submit(fetch_document, api_key, doc_id): doc_id for doc_id in new_document_ids}
            for future in as_completed(future_to_doc):
                try:
                    documents.append(future.result())
                    dataset_documents.append(future.result())
                    if len(documents) >= N:
                        break
                except Exception as e:
                    print(f"An error occurred: {e}")
        
        # To avoid hitting the API rate limit
        time.sleep(0.1)  # Adjust sleep time as needed
        
        if len(documents) >= N:
            break

    if len(documents) >= N:
        break

In [ ]:
# Check the number of documents collected
print(f'Total documents collected: {len(documents)}')

In [ ]:
def save_documents(dataset_name, documents):
    if not os.path.exists(dataset_name):
        os.makedirs(dataset_name)

    for i, doc in enumerate(documents):
        with open(f'{dataset_name}/document_{i}.rdf', 'w') as f:
            f.write(doc)

save_documents("hunnid_2", documents)

In [ ]:
# Function to get document IDs from a dataset
def get_document_ids(dataset_name, rows):
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&sort=random_1%20asc&profile=minimal'
    response = requests.get(search_url)
    data = response.json()
    document_ids = [item['id'] for item in data['items']]
    return document_ids

# Function to download documents using document IDs
def download_documents(document_ids, format='rdf'):
    documents = []
    for doc_id in document_ids:
        record_url = f'https://api.europeana.eu/record/v2/{doc_id}.{format}?wskey={api_key}'
        response = requests.get(record_url)
        documents.append(response.text)
    return documents

In [ ]:
import os

def download_documents(document_ids, format='json'):
    documents = []
    folder_path = 'collection_experiment'
    os.makedirs(folder_path, exist_ok=True)  # Create the folder if it doesn't exist
    for doc_id in document_ids:
        record_url = f'https://api.europeana.eu/record/v2/{doc_id}.{format}?wskey={api_key}'
        response = requests.get(record_url)
        file_path = os.path.join(folder_path, f'{doc_id}.{format}')
        with open(file_path, 'w') as file:
            file.write(response.text)
        documents.append(file_path)
    return documents